# XGBoost — Base vs Google Trends

**Part 1** trains and evaluates XGBoost on price + engineered features.  
**Part 2** adds 5 Google Trends features and evaluates independently.  
**Part 3** compares both models head-to-head.

Both models use market-level sample weighting (`1 / n_snapshots`) and early stopping on a held-out validation set.

In [12]:
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    f1_score, log_loss, precision_recall_curve, roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

In [13]:
ROOT            = pathlib.Path("../..")
DATA_DIR        = ROOT / "data"
ARTIFACTS_DIR   = pathlib.Path("artifacts")
PREDICTIONS_DIR = pathlib.Path("predictions")

NUMERIC_FEATURES = [
    "price_at_snapshot",
    "price_deviation_from_half",
    "days_before_close",
    "pct_lifetime_elapsed",
    "duration_days",
    "log_volume",
    "price_mean_7d",   "price_volatility_7d",  "price_min_7d",  "price_max_7d",
    "price_change_7d", "price_range_7d",        "price_trend_7d",
    "price_mean_14d",  "price_volatility_14d", "price_min_14d", "price_max_14d",
    "price_change_14d","price_range_14d",       "price_trend_14d",
]
TRENDS_FEATURES      = ["trend_value", "trend_ma4", "trend_change_4w", "trend_spike", "has_trend_data"]
CATEGORICAL_FEATURES = ["category"]
TARGET               = "outcome"

FEATURES_BASE   = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FEATURES_TRENDS = NUMERIC_FEATURES + TRENDS_FEATURES + CATEGORICAL_FEATURES

---
## Load Data

In [14]:
df = pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_clean.parquet")

df["category"] = df["category"].fillna("other")
df = df.dropna(subset=[TARGET])

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

assert len(set(train["market_id"]) & set(test["market_id"])) == 0, "Market leakage detected"

# Validation split for early stopping (15% of train markets, held out at market level)
rng            = np.random.default_rng(42)
all_market_ids = train["market_id"].unique()
val_ids        = set(rng.choice(all_market_ids, size=int(len(all_market_ids) * 0.15), replace=False))

train_fit = train[~train["market_id"].isin(val_ids)]
train_val = train[ train["market_id"].isin(val_ids)]
test_reset = test.reset_index(drop=True)

# Market-level sample weights for train_fit
counts         = train_fit.groupby("market_id").size()
sample_weights = train_fit["market_id"].map(counts).rdiv(1).values

y_fit  = train_fit[TARGET].values
y_val  = train_val[TARGET].values
y_test = test[TARGET].values

scale_pos_weight = float((y_fit == 0).sum() / (y_fit == 1).sum())

print(f"Total rows : {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Train fit  : {len(train_fit):,}  |  {train_fit['market_id'].nunique():,} markets")
print(f"Train val  : {len(train_val):,}   |  {train_val['market_id'].nunique():,} markets")
print(f"Test       : {len(test):,}   |  {test['market_id'].nunique():,} markets")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
df.head()

Total rows : 1,448,142  |  Columns: 32
Train fit  : 987,929  |  14,258 markets
Train val  : 171,723   |  2,516 markets
Test       : 288,490   |  4,174 markets
scale_pos_weight: 3.51


,market_id,snapshot_timestamp,days_before_close,pct_lifetime_elapsed,duration_days,price_at_snapshot,price_deviation_from_half,total_volume,log_volume,outcome,...,price_range_14d,price_trend_14d,split,category,question,trend_value,trend_ma4,trend_change_4w,trend_spike,has_trend_data
0,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-01 18:45:42.437000+00:00,59.22,0.1912,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000526,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
1,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 06:45:42.437000+00:00,58.72,0.1980,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000354,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
2,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 18:45:42.437000+00:00,58.22,0.2049,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000215,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
3,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 06:45:42.437000+00:00,57.72,0.2117,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000479,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
4,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 18:45:42.437000+00:00,57.22,0.2185,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000499,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1


---
## Shared Helpers

In [15]:
def build_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="constant", fill_value=0), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

def build_xgb():
    return XGBClassifier(
        n_estimators=1000,
        early_stopping_rounds=50,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

def get_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thresh[np.argmax(f1)]), float(np.max(f1))

def evaluate(y_true, y_prob, label, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "AUC-ROC" : roc_auc_score(y_true, y_prob),
        "PR-AUC"  : average_precision_score(y_true, y_prob),
        "Log-loss": log_loss(y_true, y_prob),
        "Brier"   : brier_score_loss(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1"      : f1_score(y_true, y_pred),
    }
    print(f"\n{'─'*50}")
    print(f"  {label}  (threshold={threshold:.3f})")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<10}: {val:.4f}")
    print(f"{'─'*50}")
    return metrics

def per_category(test_df, y_prob, threshold):
    rows = []
    for cat in sorted(test_df["category"].unique()):
        mask = test_df["category"] == cat
        yt   = test_df.loc[mask, TARGET].values
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        yp = y_prob[mask.values]
        rows.append({
            "category": cat,
            "n"       : int(mask.sum()),
            "AUC"     : roc_auc_score(yt, yp),
            "PR-AUC"  : average_precision_score(yt, yp),
            "F1"      : f1_score(yt, (yp >= threshold).astype(int)),
            "YES%"    : float(yt.mean()),
        })
    return pd.DataFrame(rows).set_index("category").sort_values("AUC", ascending=False)

def market_eval(test_df, y_prob, threshold):
    mdf = (
        test_df.assign(pred_prob=y_prob)
        .groupby("market_id")
        .agg(pred_prob=("pred_prob", "mean"), outcome=(TARGET, "first"))
        .reset_index()
    )
    mp, mt = mdf["pred_prob"].values, mdf["outcome"].values
    mpred  = (mp >= threshold).astype(int)
    print(f"Market-level evaluation ({len(mdf):,} markets)")
    print(f"  AUC-ROC  : {roc_auc_score(mt, mp):.4f}")
    print(f"  PR-AUC   : {average_precision_score(mt, mp):.4f}")
    print(f"  Brier    : {brier_score_loss(mt, mp):.4f}")
    print(f"  Accuracy : {accuracy_score(mt, mpred):.4f}")
    print(f"  F1       : {f1_score(mt, mpred):.4f}")
    return {
        "AUC-ROC" : roc_auc_score(mt, mp),
        "PR-AUC"  : average_precision_score(mt, mp),
        "Brier"   : brier_score_loss(mt, mp),
        "Accuracy": accuracy_score(mt, mpred),
        "F1"      : f1_score(mt, mpred),
    }

def feature_importance_df(preprocessor, clf, num_cols, trend_cols=None):
    cat_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
    all_names = num_cols + cat_names
    return (
        pd.DataFrame({"feature": all_names, "importance": clf.feature_importances_})
        .assign(is_trend=lambda d: d["feature"].isin(trend_cols or []))
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

---
## Baseline — Market Price

The simplest predictor: use `price_at_snapshot` directly as the probability.
This is the crowd's consensus — a useful sanity check for any model we build.

In [16]:
y_prob_baseline        = test_reset["price_at_snapshot"].values
thresh_bl, f1_bl       = get_threshold(y_test, y_prob_baseline)
print(f"Baseline threshold: {thresh_bl:.3f}  |  F1: {f1_bl:.4f}")

metrics_baseline_row = evaluate(y_test, y_prob_baseline, "Baseline (market price)", thresh_bl)
metrics_baseline_mkt = market_eval(test_reset, y_prob_baseline, thresh_bl)
cat_baseline         = per_category(test_reset, y_prob_baseline, thresh_bl)

Baseline threshold: 0.500  |  F1: 0.6699

──────────────────────────────────────────────────
  Baseline (market price)  (threshold=0.500)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8890
  PR-AUC    : 0.7440
  Log-loss  : 0.3278
  Brier     : 0.1004
  Accuracy  : 0.8703
  F1        : 0.6699
──────────────────────────────────────────────────
Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9420
  PR-AUC   : 0.8283
  Brier    : 0.0652
  Accuracy : 0.9178
  F1       : 0.7351


---
# Part 1 — Base Model

Price + engineered features only, no Google Trends.

## 1.1 Train

In [17]:
prep_base  = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
X_fit_base = prep_base.fit_transform(train_fit[FEATURES_BASE])
X_val_base = prep_base.transform(train_val[FEATURES_BASE])
X_test_base= prep_base.transform(test[FEATURES_BASE])

clf_base = build_xgb()
clf_base.fit(
    X_fit_base, y_fit,
    sample_weight=sample_weights,
    eval_set=[(X_val_base, y_val)],
    verbose=False,
)
print(f"Best iteration: {clf_base.best_iteration}  |  Best logloss: {clf_base.best_score:.4f}")

y_prob_base          = clf_base.predict_proba(X_test_base)[:, 1]
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

Best iteration: 343  |  Best logloss: 0.3557
Optimal threshold: 0.629  |  F1: 0.6785


## 1.2 Row-Level Evaluation

In [18]:
metrics_base_row = evaluate(y_test, y_prob_base, "XGBoost Base", thresh_base)


──────────────────────────────────────────────────
  XGBoost Base  (threshold=0.629)
──────────────────────────────────────────────────
  AUC-ROC   : 0.9004
  PR-AUC    : 0.7615
  Log-loss  : 0.3822
  Brier     : 0.1206
  Accuracy  : 0.8587
  F1        : 0.6785
──────────────────────────────────────────────────


## 1.3 Per-Category Breakdown

In [19]:
cat_base = per_category(test_reset, y_prob_base, thresh_base)
cat_base.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9306,0.7371,0.6157,13.8%
entertainment,37027,0.9226,0.7340,0.6249,14.9%
crypto,24586,0.9207,0.8178,0.7607,26.1%
politics_us,52699,0.9174,0.8260,0.7284,24.7%
politics_global,22591,0.9149,0.7648,0.7216,22.3%
finance,25093,0.9086,0.8095,0.6938,24.3%
science_tech,15530,0.9026,0.7554,0.6465,17.8%
sports,93022,0.8711,0.7069,0.6287,20.9%
other,1364,0.8671,0.3352,0.3401,4.7%


## 1.4 Market-Level Evaluation

In [20]:
metrics_base_mkt = market_eval(test_reset, y_prob_base, thresh_base)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9437
  PR-AUC   : 0.8312
  Brier    : 0.0774
  Accuracy : 0.9082
  F1       : 0.7349


## 1.5 Feature Importance

In [21]:
imp_base = feature_importance_df(prep_base, clf_base, NUMERIC_FEATURES)
imp_base.head(20).style.bar(subset=["importance"], color="#5fba7d").format({"importance": "{:.4f}"})

,feature,importance,is_trend
0,price_at_snapshot,0.4686,False
1,price_mean_7d,0.2670,False
2,price_deviation_from_half,0.0531,False
3,price_min_7d,0.0333,False
4,price_max_7d,0.0124,False
5,log_volume,0.0116,False
6,price_max_14d,0.0107,False
7,category_sports,0.0097,False
8,category_science_tech,0.0091,False
9,category_geopolitics,0.0087,False


---
# Part 2 — Model with Google Trends

Same as Part 1 plus 5 Google Trends features: `trend_value`, `trend_ma4`, `trend_change_4w`, `trend_spike`, `has_trend_data`.

## 2.1 Train

In [22]:
prep_trends   = build_preprocessor(NUMERIC_FEATURES + TRENDS_FEATURES, CATEGORICAL_FEATURES)
X_fit_trends  = prep_trends.fit_transform(train_fit[FEATURES_TRENDS])
X_val_trends  = prep_trends.transform(train_val[FEATURES_TRENDS])
X_test_trends = prep_trends.transform(test[FEATURES_TRENDS])

clf_trends = build_xgb()
clf_trends.fit(
    X_fit_trends, y_fit,
    sample_weight=sample_weights,
    eval_set=[(X_val_trends, y_val)],
    verbose=False,
)
print(f"Best iteration: {clf_trends.best_iteration}  |  Best logloss: {clf_trends.best_score:.4f}")

y_prob_trends            = clf_trends.predict_proba(X_test_trends)[:, 1]
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

Best iteration: 779  |  Best logloss: 0.3504
Optimal threshold: 0.626  |  F1: 0.6769


## 2.2 Row-Level Evaluation

In [23]:
metrics_trends_row = evaluate(y_test, y_prob_trends, "XGBoost Trends", thresh_trends)


──────────────────────────────────────────────────
  XGBoost Trends  (threshold=0.626)
──────────────────────────────────────────────────
  AUC-ROC   : 0.9002
  PR-AUC    : 0.7608
  Log-loss  : 0.3750
  Brier     : 0.1181
  Accuracy  : 0.8574
  F1        : 0.6769
──────────────────────────────────────────────────


## 2.3 Per-Category Breakdown

In [24]:
cat_trends = per_category(test_reset, y_prob_trends, thresh_trends)
cat_trends.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9339,0.7407,0.6340,13.8%
crypto,24586,0.9273,0.8314,0.7713,26.1%
entertainment,37027,0.9243,0.7347,0.6479,14.9%
politics_us,52699,0.9148,0.8245,0.7300,24.7%
finance,25093,0.9059,0.8068,0.6901,24.3%
politics_global,22591,0.8978,0.7311,0.6701,22.3%
science_tech,15530,0.8974,0.7461,0.6282,17.8%
sports,93022,0.8722,0.7064,0.6277,20.9%
other,1364,0.8588,0.3053,0.3115,4.7%


## 2.4 Market-Level Evaluation

In [25]:
metrics_trends_mkt = market_eval(test_reset, y_prob_trends, thresh_trends)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9429
  PR-AUC   : 0.8294
  Brier    : 0.0751
  Accuracy : 0.9097
  F1       : 0.7362


## 2.5 Feature Importance

Trend features are highlighted in yellow.

In [26]:
imp_trends = feature_importance_df(prep_trends, clf_trends, NUMERIC_FEATURES + TRENDS_FEATURES, TRENDS_FEATURES)

print("Trend feature importances:")
print(imp_trends[imp_trends["is_trend"]].to_string(index=False))
print()

imp_trends.head(25).style.bar(
    subset=["importance"], color="#5fba7d"
).apply(
    lambda col: ["background-color: #fff3cd" if v else "" for v in imp_trends.head(25)["is_trend"]],
    axis=0, subset=["feature", "importance"]
).format({"importance": "{:.4f}"})

Trend feature importances:
        feature  importance  is_trend
      trend_ma4    0.007992      True
trend_change_4w    0.007758      True
    trend_value    0.007315      True
    trend_spike    0.002931      True
 has_trend_data    0.000000      True



,feature,importance,is_trend
0,price_at_snapshot,0.4306,False
1,price_mean_7d,0.2410,False
2,price_min_7d,0.0608,False
3,price_deviation_from_half,0.0485,False
4,log_volume,0.0116,False
5,price_max_7d,0.0111,False
6,price_change_7d,0.0108,False
7,category_sports,0.0108,False
8,category_crypto,0.0104,False
9,category_entertainment,0.0102,False


---
# Part 3 — Comparison

Head-to-head: Base vs Trends across row-level metrics, market-level metrics, and per-category AUC.

## 3.1 Row-Level

In [27]:
row_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_row,
    "Base":     metrics_base_row,
    "Trends":   metrics_trends_row,
})
row_comparison["Δ Base"]   = row_comparison["Base"]   - row_comparison["Baseline"]
row_comparison["Δ Trends"] = row_comparison["Trends"] - row_comparison["Baseline"]
row_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.8890,0.9004,0.9002,0.0113,0.0112
PR-AUC,0.7440,0.7615,0.7608,0.0175,0.0168
Log-loss,0.3278,0.3822,0.3750,0.0544,0.0472
Brier,0.1004,0.1206,0.1181,0.0201,0.0177
Accuracy,0.8703,0.8587,0.8574,-0.0116,-0.0129
F1,0.6699,0.6785,0.6769,0.0087,0.0071


## 3.2 Market-Level

In [28]:
mkt_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_mkt,
    "Base":     metrics_base_mkt,
    "Trends":   metrics_trends_mkt,
})
mkt_comparison["Δ Base"]   = mkt_comparison["Base"]   - mkt_comparison["Baseline"]
mkt_comparison["Δ Trends"] = mkt_comparison["Trends"] - mkt_comparison["Baseline"]
mkt_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.9420,0.9437,0.9429,0.0017,0.0009
PR-AUC,0.8283,0.8312,0.8294,0.0029,0.0011
Brier,0.0652,0.0774,0.0751,0.0122,0.0099
Accuracy,0.9178,0.9082,0.9097,-0.0096,-0.0081
F1,0.7351,0.7349,0.7362,-0.0002,0.0010


## 3.3 Per-Category AUC Delta

In [29]:
cat_comparison = cat_baseline[["n", "AUC", "YES%"]].rename(columns={"AUC": "AUC (baseline)"})
cat_comparison["AUC (base)"]   = cat_base["AUC"]
cat_comparison["AUC (trends)"] = cat_trends["AUC"]
cat_comparison["Δ Base"]       = cat_comparison["AUC (base)"]   - cat_comparison["AUC (baseline)"]
cat_comparison["Δ Trends"]     = cat_comparison["AUC (trends)"] - cat_comparison["AUC (baseline)"]
(
    cat_comparison
    .sort_values("AUC (baseline)", ascending=False)
    .style
    .format("{:.4f}", subset=["AUC (baseline)", "AUC (base)", "AUC (trends)", "Δ Base", "Δ Trends"])
    .format("{:.1%}", subset=["YES%"])
    .bar(subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"])
)

,n,AUC (baseline),YES%,AUC (base),AUC (trends),Δ Base,Δ Trends
category,,,,,,,
geopolitics,16578,0.9265,13.8%,0.9306,0.9339,0.0041,0.0074
finance,25093,0.9183,24.3%,0.9086,0.9059,-0.0097,-0.0124
entertainment,37027,0.9101,14.9%,0.9226,0.9243,0.0125,0.0142
politics_us,52699,0.9083,24.7%,0.9174,0.9148,0.0091,0.0064
politics_global,22591,0.9057,22.3%,0.9149,0.8978,0.0092,-0.0079
crypto,24586,0.9052,26.1%,0.9207,0.9273,0.0155,0.0220
science_tech,15530,0.8876,17.8%,0.9026,0.8974,0.0150,0.0098
other,1364,0.8847,4.7%,0.8671,0.8588,-0.0176,-0.0259
sports,93022,0.8400,20.9%,0.8711,0.8722,0.0311,0.0322


---
## Save Artifacts

In [30]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"preprocessor": prep_base,   "clf": clf_base},   ARTIFACTS_DIR / "model_base.joblib")
joblib.dump({"preprocessor": prep_trends, "clf": clf_trends}, ARTIFACTS_DIR / "model_trends.joblib")
print(f"Models saved → {ARTIFACTS_DIR}")

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_df = test[["market_id", "category", TARGET]].copy().reset_index(drop=True)
pred_df["pred_prob_base"]    = y_prob_base
pred_df["pred_prob_trends"]  = y_prob_trends
pred_df["pred_label_base"]   = (y_prob_base   >= thresh_base).astype(int)
pred_df["pred_label_trends"] = (y_prob_trends >= thresh_trends).astype(int)
preds_path = PREDICTIONS_DIR / "predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")
pred_df.head()

Models saved → artifacts
Predictions saved → predictions/predictions.csv


,market_id,category,outcome,pred_prob_base,pred_prob_trends,pred_label_base,pred_label_trends
0,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.695294,0.529934,1,0
1,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.781377,0.622801,1,0
2,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.824547,0.677071,1,1
3,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.797577,0.624029,1,0
4,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.915922,0.825984,1,1
